In [1]:
from content.connection.credentials import read_credentials
from content.connection.connect_db import create_conection
from content.extraction.read_db import read_file
from content.db.querys import read_asignation_naturgy
import os
import shutil
import pandas as pd
import re
from datetime import datetime
from psycopg2.extras import execute_batch
import psycopg2

In [2]:
credentials = read_credentials()
engine_local = create_conection(credentials)

Conexion exitosa a la DB


# EXTRACTION
## Lectura de archivos 

* Se realiza la lectura de las dos carpetas, el repositorio que contiene el historico de archivos recibidos y la carpeta donde se esta dejando todo lo nuevo


In [3]:
path_new = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Nueva_asignacion'
path_old = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Repositorio'

#df_mail_repository, df_sinfin_repository = read_file(path_old)
df_mail_new, df_sinfin_new = read_file(path_new)

Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Nueva_asignacion


PermissionError: [Errno 13] Permission denied: 'Z:\\1. Coordinadores\\Asignaciones\\Naturgy\\Bases\\Nueva_asignacion\\~$21_08_2024.xlsx'

# TRANSFORM

In [ ]:
#print(len(df_sinfin_repository))
print(len(df_sinfin_new))

Se ordena los df para eliminar registros duplicados, dejando solo la asignación mas resiente, se agrega en el df resultado el email que viene originalmente en otra pestaña

In [ ]:
def join_df_mail_sinfin(df_mail, df_sifin):
    df_mail = df_mail.sort_values(by=['CTA CONTR', 'FECHA DE CESION'], ascending=[True, False])
    df_mail = df_mail.drop_duplicates(subset=['CTA CONTR'])
    df_sifin = df_sifin.sort_values(by=['NUMERO_CUENTA', 'DATE1'], ascending=[True, False])
    df_sifin = df_sifin.drop_duplicates(subset=['NUMERO_CUENTA'])
    df_mail = df_mail[['CTA CONTR', 'CORREO ELECTRONICO']]
    df_repository = pd.merge(df_mail, df_sifin, how='right', left_on='CTA CONTR', right_on='NUMERO_CUENTA')

    df_repository['EmailsDeudor'] = df_repository['CORREO ELECTRONICO']

    df_repository = df_repository.drop(columns=['CTA CONTR', 'CORREO ELECTRONICO'])

    return df_repository
    

In [ ]:
#df_consolidated_old = join_df_mail_sinfin(df_mail_repository, df_sinfin_repository)
df_consolidated_new = join_df_mail_sinfin(df_mail_new, df_sinfin_new)

In [ ]:
df_group_read=df_consolidated_new.groupby(['name_file', 'TEXT6']).agg(
    Q = ('PRIMER_NOMBRE', 'count')
)

df_group_read['Q'] = df_group_read['Q'].astype(str)
df_group_read = df_group_read.sort_values(by='name_file', ascending=True)
print(df_group_read)

In [ ]:
#df_consolidated_old.to_sql('asignacion', engine_local, if_exists='append', index=False, schema='sinfin')

In [ ]:
text_query = read_asignation_naturgy()

df_asignation_db = pd.read_sql_query(text_query, engine_local)
print(len(df_asignation_db))


In [ ]:
df_row_asignated = pd.merge(df_asignation_db, df_consolidated_new, on='NUMERO_CUENTA', how='inner')

In [ ]:
columns_diference = df_row_asignated[['NUMERO_CUENTA', 'name_file_x','TEXT8_x', 'TEXT6_x', 'MONEY1_x', 'DATE1_x', 'DATE2_x', 'TEXT8_y', 'TEXT6_y','MONEY1_y', 'DATE1_y', 'DATE2_y', 'name_file_y']]
columns_diference['DIAS'] = (columns_diference['DATE1_y'] - columns_diference['DATE2_x']).dt.days

columns_diference['MES'] = columns_diference['DATE2_x'].dt.month
columns_diference = columns_diference[columns_diference['DIAS'] < 0]
columns_diference = columns_diference[columns_diference['MES'] >= 8]

columns_diference['NUMERO_CUENTA'] = columns_diference['NUMERO_CUENTA'].astype(str)
def validate_file(row):
    
    if row['name_file_x'] == row['name_file_y']:
        return 'Si'
    else:
        return 'No'

columns_diference['Val']=columns_diference.apply(validate_file, axis=1)
columns_diference = columns_diference[columns_diference['Val'] == 'No']
 


In [ ]:
print(len(df_row_asignated))

In [ ]:
df_data_update = pd.merge(df_consolidated_new, df_asignation_db, on='NUMERO_CUENTA')

len(df_data_update)

In [ ]:
df_new_data = pd.merge(df_consolidated_new, df_asignation_db, on='NUMERO_CUENTA', how='left', indicator=True)
df_new_data = df_new_data[df_new_data['_merge']== 'left_only']
df_new_data = df_new_data[df_new_data['_merge'] == 'left_only'].drop(columns=['_merge'])
print(len(df_new_data))

In [ ]:
def clear_df(df):
    columns_drop = [col for col in df.columns if col.endswith('_y')]
    df = df.drop(columns=columns_drop)
    df.columns = [col.replace('_x','') for col in df.columns]
    df = df.reset_index()
    df['TEXT10'] = pd.NA
    
    df = df.dropna(subset=['NUMERO_CUENTA'])
    try:
        df = df.drop(columns=['Dirección'])
    except:
        print('')
        
    try:
          df = df.drop(columns=['index'])  
    except:
        print('')
        
    print(list(df.columns))
    print(len(df))
    
    return df

In [ ]:
df_new_data = clear_df(df_new_data)
df_data_update = clear_df(df_data_update)

In [ ]:
df_new_data['NUMERO_CUENTA'] = pd.to_numeric(df_new_data['NUMERO_CUENTA'], errors='coerce')

# load
## Actualizar filas con la información de la nueva asignación

In [ ]:

con = psycopg2.connect("dbname=Estrategia user=CDM password=password host=localhost port=5432")
cur = con.cursor()
data = df_data_update[[
    'MONEY1',
    'MONEY4',
    'DATE1',
    'DATE2',
    'TEXT1',
    'TEXT2',
    'TEXT3',
    'TEXT4',
    'TEXT5',
    'TEXT6',
    'TEXT7',
    'TEXT8',
    'TEXT9',
    'name_file',
    'EmailsDeudor',
    'NUMERO_CUENTA'

]].values.tolist()
# Ejecutar las actualizaciones en bloque
execute_batch(cur, """
    UPDATE sinfin.asignacion
    SET "MONEY1" = %s,
        "MONEY4" = %s,
        "DATE1" = %s,
        "DATE2" = %s,
        "TEXT1" = %s,
        "TEXT2" = %s,
        "TEXT3" = %s,
        "TEXT4" = %s,
        "TEXT5" = %s,
        "TEXT6" = %s,
        "TEXT7" = %s,
        "TEXT8" = %s,
        "TEXT9" = %s,
        "name_file" = %s,
        "EmailsDeudor" =%s
    WHERE "NUMERO_CUENTA" = %s
""", data)

con.commit()
cur.close()
con.close()

# Cargar nuevas filas de la asignación

In [ ]:
df_new_data.to_sql('asignacion', engine_local, if_exists='append', index=False, schema='sinfin')

In [ ]:
#path_new
list_file_new_asignation = os.listdir(path_new)
for name in list_file_new_asignation:
    name_file_source = os.path.join(path_new, name)
    name_file_destination = os.path.join(path_old, name)
    shutil.move(name_file_source, name_file_destination )
    
print(f'Se realizo el movimiento de {len(list_file_new_asignation)} archivo a la carpeta del repositorio.')

# Transform 
## DF


In [14]:
def transform_df(df):
    date_today = datetime.today()
    df['DATE1'] = pd.to_datetime(df['DATE1'], errors='coerce')
    df['TEXT6'] = df['TEXT6'].apply(lambda x: x.strip() if isinstance(x, str) else x)
    df['MONEY1'] = df['MONEY1'].round(2)
    df['MONEY2'] = df['MONEY2'].round(2)
    df['MONEY3'] = df['MONEY3'].round(2)
    df['MONEY4'] = df['MONEY4'].round(2)
    

    def change_typedata(df):
        column_convert = ['NUMERO_CUENTA', 'TEXT2', 'TEXT4', 'TEXT6', 'TEXT9', 'Barrio1Deudor', 'PRODUCTO_ID']
        for col in column_convert:
            if col in df.columns:
                df[col] = df[col].astype(str).apply(lambda x: re.sub(r'\.0$', '', x))
            else:
                raise KeyError(f"La columna {col} no existe en el DataFrame")
        
        
        df['DATE1'] = pd.to_datetime(df['DATE1'], errors='coerce')
        df['DATE2'] = pd.to_datetime(df['DATE2'], errors='coerce')
        
        df['PRODUCTO_ID'] = df['TEXT6']

        
        return df
    
    def filter_estatus_date(df):
        df['TEXT20'] = df['DATE2'].apply(lambda x: 'Retirado' if x < date_today else 'Vigente')
        df = df[df['TEXT20'] == 'Vigente']
        
        return df

    
    df = change_typedata(df)
    df = filter_estatus_date(df)

    new_order_columns = ['IDENTIFICACION', 'TIPO_DOC', 'PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'SEXO', 'ESTADO_CIVIL', 'PERSONAS_A_CARGO', 'FECHA_NAC', 'IDIOMA', 'EMPRESA', 'CARGO', 'PROFESION', 'TIPO_VIVIENDA', 'NUMERO_CUENTA', 'PRODUCTO_ID', 'CLIENTE_ID', 'CUENTA_CLIENTE_ID', 'TEXT1', 'TEXT2', 'TEXT3', 'TEXT4', 'TEXT5', 'TEXT6', 'TEXT7', 'TEXT8', 'TEXT9', 'TEXT10', 'TEXT11', 'TEXT12', 'TEXT13', 'TEXT14', 'TEXT15', 'TEXT16', 'TEXT17', 'TEXT18', 'TEXT19', 'TEXT20', 'MONEY1', 'MONEY2', 'MONEY3', 'MONEY4', 'MONEY5', 'MONEY6', 'MONEY7', 'MONEY8', 'MONEY9', 'MONEY10', 'MONEY11', 'MONEY12', 'MONEY13', 'MONEY14', 'MONEY15', 'MONEY16', 'MONEY17', 'MONEY18', 'MONEY19', 'MONEY20', 'NUMBER1', 'NUMBER2', 'NUMBER3', 'NUMBER4', 'NUMBER5', 'PERCENT1', 'PERCENT2', 'PERCENT3', 'DATE1', 'DATE2', 'DATE3', 'DATE4', 'DATE5', 'DATE6', 'DATE7', 'Dir1Deudor', 'Ciudad1Deudor', 'Dpto1Deudor', 'Barrio1Deudor', 'TelesDeudor', 'Dir2Deudor', 'Ciudad2Deudor', 'Dpto2Deudor', 'Barrio2Deudor', 'EmailsDeudor', 'DirEmpDeudor', 'CiudadEmpDeudor', 'DptoEmpDeudor', 'TelesEmpDeudor', 'IdentConyuge', 'Nombrecy', 'Dir1cy', 'Ciudad1cy', 'Dpto1cy', 'Telescy', 'Emailscy', 'IdentCodeudor1', 'NombreCodeudor1', 'Dir1Codeudor1', 'Ciudad1Codeudor1', 'Dpto1Codeudor1', 'TelesCodeudor1', 'EmailsCodeudor1', 'IdentRef1', 'NombreRef1', 'Dir1Ref1', 'Ciudad1Ref1', 'Dpto1Ref1', 'TelesRef1', 'EmailsRef1', 'IdentRef2', 'NombreRef2', 'Dir1Ref2', 'Ciudad1Ref2', 'Dpto1Ref2', 'TelesRef2', 'EmailsRef2','sucursal_responsable', 'ejecutivo_responsable', 'date_entered']
    df = df.reindex(columns = new_order_columns)
    
    return df

In [12]:
text_query = read_asignation_naturgy()

df_data_total_asignation = pd.read_sql_query(text_query, engine_local)
print(len(df_data_total_asignation))


135999


In [15]:
df_data_total_asignation = transform_df(df_data_total_asignation)

In [17]:
def estado_pago(row):
    if row['fecha_pago'] >= row['DATE1'] <= row['DATE2']:
        return 'Si'
    else:
        return 'No'

In [18]:
def aplicated_payment(row):
    
    if row['payment'] > 0 and row['estado_pago'] == 'Si':
        reclamado = row['payment']
        pendiente = round(row['MONEY1'] - row['payment'],2)
        return pd.Series([reclamado, pendiente], index=['reclamado', 'pendiente'])
    else:
        return pd.Series([0, row['MONEY4']], index=['reclamado', 'pendiente'])

In [19]:
def update_data_db():
    text_dic = """
        SELECT      *
        FROM        sinfin.dic_naturgy
    """
    df_dic_naturgy = pd.read_sql_query(text_dic, engine_local)

    text_payment = """
        SELECT      date_entered as fecha_pago,  
                    "ACCOUNT_NUMBER",
                    "PAYMENT_AMOUNT"
        FROM        data_sinfin.pagos
        WHERE       "ENTIDAD_ID" = 'NATURGY'
    """

    df_payments = pd.read_sql_query(text_payment, engine_local)

    df_total_saldo = pd.merge (df_data_total_asignation, df_dic_naturgy, left_on='TEXT8', right_on= 'Vuelta', how='right')
    df_total_saldo['TEXT8'] = df_total_saldo['Vuelta_correcta']
    df_total_saldo['TEXT2'] = df_total_saldo['Tipo']
    #df_total_saldo['NUMERO_CUENTA'] = df_total_saldo['NUMERO_CUENTA'].astype(str)

    df_payment_sum = df_payments.groupby(['ACCOUNT_NUMBER', 'fecha_pago']).agg(
        payment = ('PAYMENT_AMOUNT', 'sum')
        ).reset_index()

    print(df_payment_sum )

    df_asignation_payments_aplicated = pd.merge(df_total_saldo, df_payment_sum, left_on='NUMERO_CUENTA', right_on='ACCOUNT_NUMBER', indicator=True, how='left')

    df_asignation_payments_aplicated['estado_pago'] = df_asignation_payments_aplicated.apply(estado_pago, axis=1)
    df_asignation_payments2 = df_asignation_payments_aplicated[df_asignation_payments_aplicated['estado_pago'] == 'Si']


    df_asignation_payments_aplicated[['MONEY2', 'MONEY4']] = df_asignation_payments_aplicated.apply(aplicated_payment, axis=1)
    df_asignation_payments_aplicated=df_asignation_payments_aplicated.drop(columns=['payment', '_merge'])
    #df_asignation_payments_aplicated = df_asignation_payments_aplicated[df_asignation_payments_aplicated['MONEY4'] > 0]


    print(len(df_asignation_payments_aplicated))
    df_upload_sinfin =df_asignation_payments_aplicated[df_asignation_payments_aplicated['sucursal_responsable'] == 'Colombia']

    print(len(df_upload_sinfin ))
    
    return df_asignation_payments_aplicated

In [22]:
from itertools import cycle

def separate_df(df):
    print(list(df.columns))
    df['date_entered'] = datetime.now().strftime('%Y/%m/%d')
    df = df.drop_duplicates(subset=['NUMERO_CUENTA'], keep='first')

    def sort_df(df):
        df = df.sort_values(by=['DATE1', 'MONEY1'], ascending=True)
        return df
    
    def procesing_data_upload(df_updata):
        list_data = df_updata[[
        'sucursal_responsable',
        'date_entered',
        'ejecutivo_responsable',
        'NUMERO_CUENTA'
        ]].values.tolist()

        text_update = """
                UPDATE sinfin.asignacion
                SET "sucursal_responsable" = %s,
                    "date_entered" = %s,
                    "ejecutivo_responsable" = %s
                WHERE "NUMERO_CUENTA" = %s
                """
        return list_data, text_update
    
    
    def update_data_sql(df_data, text_update):
        con = psycopg2.connect("dbname=Estrategia user=CDM password=password host=localhost port=5432")
        cur = con.cursor()
        data = df_data
        # Ejecutar las actualizaciones en bloque
        execute_batch(cur, text_update, data)

        con.commit()
        cur.close()
        con.close()
    
    df_colombia_baja = df[(df['TEXT2'] == 'BAJA - VENCIDA') & (df['sucursal_responsable'] == 'Colombia')]
    df_colombia_activa = df[(df['TEXT2'] == 'ACTIVA - VIGENTE') & (df['sucursal_responsable'] == 'Colombia')]
    df_anonimous_baja = df[(df['TEXT2'] == 'BAJA - VENCIDA') & (df['sucursal_responsable'].isna())]
    df_anonimous_activa = df[(df['TEXT2'] == 'ACTIVA - VIGENTE') & (df['sucursal_responsable'].isna())]
    print(f'Cantidad de filas para entregar entre oficinas: {len(df_anonimous_activa)}')
    
    df_colombia_baja  =sort_df(df_colombia_baja)
    df_colombia_activa =sort_df(df_colombia_activa)
    df_anonimous_baja =sort_df(df_anonimous_baja)
    df_anonimous_activa =sort_df(df_anonimous_activa)
    
    
    max_row_baja =  len(df_anonimous_baja)
    max_row_activa = len(df_anonimous_activa)
    
    list_sucursal = ['Cali', 'Bogota']
    list_sucursal_cycle = cycle(list_sucursal)
    
    df_anonimous_baja['sucursal_responsable'] =   [next(list_sucursal_cycle) for _ in range(max_row_baja)]
    df_anonimous_activa['sucursal_responsable'] =   [next(list_sucursal_cycle) for _ in range(max_row_activa)]
    
    list_data_baja, text_update_baja = procesing_data_upload(df_anonimous_baja)
    update_data_sql(list_data_baja, text_update_baja)
    
    list_data_activa, text_update_activa = procesing_data_upload(df_anonimous_activa)
    update_data_sql(list_data_activa, text_update_activa)
    
    df_baja_bogota = df_anonimous_baja[df_anonimous_baja['sucursal_responsable'] == 'Bogota']
    df_activa_bogota = df_anonimous_activa[df_anonimous_activa['sucursal_responsable'] == 'Bogota']
    
    df_baja_cali = df_anonimous_baja[df_anonimous_baja['sucursal_responsable'] == 'Cali']
    df_activa_cali = df_anonimous_activa[df_anonimous_activa['sucursal_responsable'] == 'Cali']   
    
    path_save = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo'
    df_baja_bogota.to_excel(os.path.join(path_save, 'baja_bogota.xlsx'), index=False)
    df_activa_bogota.to_excel(os.path.join(path_save, 'activa_bogota.xlsx'), index=False)
    df_baja_cali.to_excel(os.path.join(path_save, 'baja_cali.xlsx'), index=False)
    df_activa_cali.to_excel(os.path.join(path_save, 'activa_cali.xlsx'), index=False)
    df_colombia_baja.to_excel(os.path.join(path_save, 'baja_colombia.xlsx'), index=False)
    df_colombia_activa.to_excel(os.path.join(path_save, 'activa_colombia.xlsx'), index=False)
    
    return 
  

In [23]:
separate_df(update_data_db())

      ACCOUNT_NUMBER              fecha_pago  payment
0         1000245580 2024-06-24 02:51:58.407   121.13
1           10002967 2024-05-14 05:40:21.147    22.67
2           10002967 2024-05-14 05:40:21.150    22.67
3         1000325590 2024-02-24 04:47:19.150   368.89
4         1001465568 2024-04-03 07:20:01.047    42.67
...              ...                     ...      ...
12624      995865594 2024-04-03 07:20:01.047    33.60
12625      995865594 2024-06-24 02:51:58.400    34.11
12626      995865594 2024-06-24 02:51:58.417    22.45
12627      997625585 2024-03-15 12:06:21.690    61.93
12628      997925565 2024-06-25 11:49:34.503    84.49

[12629 rows x 3 columns]
23550
14863
['IDENTIFICACION', 'TIPO_DOC', 'PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'SEXO', 'ESTADO_CIVIL', 'PERSONAS_A_CARGO', 'FECHA_NAC', 'IDIOMA', 'EMPRESA', 'CARGO', 'PROFESION', 'TIPO_VIVIENDA', 'NUMERO_CUENTA', 'PRODUCTO_ID', 'CLIENTE_ID', 'CUENTA_CLIENTE_ID', 'TEXT1', 'TEXT2', 'TEXT3'

In [24]:
data=update_data_db()

      ACCOUNT_NUMBER              fecha_pago  payment
0         1000245580 2024-06-24 02:51:58.407   121.13
1           10002967 2024-05-14 05:40:21.147    22.67
2           10002967 2024-05-14 05:40:21.150    22.67
3         1000325590 2024-02-24 04:47:19.150   368.89
4         1001465568 2024-04-03 07:20:01.047    42.67
...              ...                     ...      ...
12624      995865594 2024-04-03 07:20:01.047    33.60
12625      995865594 2024-06-24 02:51:58.400    34.11
12626      995865594 2024-06-24 02:51:58.417    22.45
12627      997625585 2024-03-15 12:06:21.690    61.93
12628      997925565 2024-06-25 11:49:34.503    84.49

[12629 rows x 3 columns]
23550
14863


In [ ]:
def save_file(df, name):

    df['TEXT10'] = df['sucursal_responsable']
    
    print(len(df))
    df = df.sort_values(by='fecha_pago', ascending=False)
    df=df.drop(columns=['sucursal_responsable', 'ejecutivo_responsable', 'date_entered', 'index', 'Vuelta', 'Tipo', 'Vuelta_correcta', 'ACCOUNT_NUMBER', 'fecha_pago', 'estado_pago'])
    df=df.drop_duplicates(subset=['NUMERO_CUENTA'])
    print(len(df))
    
    
    path_sinfin = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Sinfin'
    name_file_sinfin = os.path.join(path_sinfin,f'{name}.xlsx')
    print(name_file_sinfin)
    df.to_excel(name_file_sinfin, index=False)
save_file(data, 'Sinfin')

In [25]:

print(len(data))
df_asignation_payments_aplicated = data.drop_duplicates(subset=['NUMERO_CUENTA'])
df_asignation_payments_aplicated = df_asignation_payments_aplicated[df_asignation_payments_aplicated['MONEY4'] > 0]
print(len(df_asignation_payments_aplicated))

df_asignation_payments_aplicated['MONEY1'] =df_asignation_payments_aplicated['MONEY1'].round(2)
df_asignation_payments_aplicated['MONEY4'] =df_asignation_payments_aplicated['MONEY4'].round(2)

df_teles = df_asignation_payments_aplicated[['NUMERO_CUENTA', 'TelesDeudor']]
df_teles_separado = df_teles['TelesDeudor'].str.split('|', expand=True)
df_teles = pd.concat([df_teles[['NUMERO_CUENTA']], df_teles_separado], axis=1)
df_teles.columns=['NUMERO_CUENTA', 'Tel1', 'Tel2', 'Tel3', 'Tel4', 'Tel5']

df_columns_predictive = df_asignation_payments_aplicated[['NUMERO_CUENTA', 'IDENTIFICACION', 'PRIMER_NOMBRE', 'TEXT1', 'TEXT2', 'TEXT6', 'MONEY1', 'MONEY4', 'DATE1', 'DATE2', 'TelesDeudor', 'sucursal_responsable', 'Dir1Deudor', 'EmailsDeudor']]
df_columns_predictive = pd.merge(df_columns_predictive, df_teles, how='left', indicator=True, on='NUMERO_CUENTA')

df_baja_bogota = df_columns_predictive[(df_columns_predictive['sucursal_responsable'] == 'Bogota') & (df_columns_predictive['TEXT2'] == 'BAJA - VENCIDA')]
df_baja_cali = df_columns_predictive[(df_columns_predictive['sucursal_responsable'] == 'Cali') & (df_columns_predictive['TEXT2'] == 'BAJA - VENCIDA')]
df_baja_colombia = df_columns_predictive[(df_columns_predictive['sucursal_responsable'] == 'Colombia') & (df_columns_predictive['TEXT2'] == 'BAJA - VENCIDA')]


df_activa_bogota = df_columns_predictive[(df_columns_predictive['sucursal_responsable'] == 'Bogota') & (df_columns_predictive['TEXT2'] == 'ACTIVA - VIGENTE')]
df_activa_cali = df_columns_predictive[(df_columns_predictive['sucursal_responsable'] == 'Cali') & (df_columns_predictive['TEXT2'] == 'ACTIVA - VIGENTE')]
df_activa_colombia = df_columns_predictive[(df_columns_predictive['sucursal_responsable'] == 'Colombia') & (df_columns_predictive['TEXT2'] == 'ACTIVA - VIGENTE')]

dic_df = {
    'baja_bogota' : df_baja_bogota,
    'baja_cali' : df_baja_cali,
    'baja_colombia' : df_baja_colombia,
    'activa_bogota' : df_activa_bogota,
    'activa_cali' : df_activa_cali,
    'activa_colombia' : df_activa_colombia
}

def save_file_predictive(df, name):
    
    path_file_predictive = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo'
    name_file = os.path.join(path_file_predictive, f'{name}.xlsx')
    df.to_excel(name_file, sheet_name='Predictivo', index= False)
    print(name_file)

for key, value in dic_df.items():
    name_file = key
    save_file_predictive(value, name_file)





23550
21921
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\baja_bogota.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\baja_cali.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\baja_colombia.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\activa_bogota.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\activa_cali.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\activa_colombia.xlsx
